# 05-01 — Seeded Cross-Site Evaluation with a Held-Out A Slice

The main 05 notebook scores the Random Forest on the same DRIAMS-A spectra it
was trained on (the "A (train)" row, BalAcc 0.999). That is training-set
memorisation, not site performance, and it must not be read as "works
perfectly at home, fails away".

This notebook re-runs the cross-site evaluation properly:

- split A into a **train slice (80%)** and a **held-out A test slice (20%)**,
  independently for five seeds;
- fit the tuned RF (500 trees, max_depth 10, min_samples_leaf 10,
  class_weight balanced) on the A-train slice only, with log1p+standardize
  fitted on that slice;
- score the **A holdout**, **B**, **C**, **D** and their union **B+C+D**;
- report two thresholds — the default **0.5** and a **cross-validated
  threshold tuned inside A-train** (3-fold out-of-fold predictions, frozen
  for every target) — plus threshold-free AUC;
- repeat the pooled 75/25 multi-seed evaluation with the same tuned-threshold
  protocol, to quantify how much pooled Balanced Accuracy is lost to the
  default threshold.

Aggregates are mean ± std over the five seeds. Outputs are written to
``results_rf/``.


In [ ]:
# =============================================================================
# 0. SETUP
# =============================================================================
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_predict, train_test_split

from maldideepkit.base.data import fit_input_transform, apply_input_transform

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    print("Google Drive mounted")
    IN_COLAB = True
except ImportError:
    print("Running locally")
    IN_COLAB = False

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    CANDIDATES = [
        Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed"),
        Path("/mnt/sda-home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
        Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed"),
    ]
    DATA_ROOT = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_ROOT is not None and DATA_ROOT.exists(), f"no data root found: {CANDIDATES}"

HERE = Path.cwd()
OUT_DIR = HERE / "results_rf"
assert OUT_DIR.exists(), f"run from the 05 notebook folder ({OUT_DIR} missing)"

DRUG, SPECIES = "Ceftazidime", "Escherichia coli"
SEEDS = [42, 123, 456, 789, 1011]
PARAMS = dict(n_estimators=500, max_depth=10, min_samples_leaf=10,
              class_weight="balanced")
THRESHOLDS = np.linspace(0.05, 0.95, 91)
print(f"DATA_ROOT: {DATA_ROOT}\nOUT_DIR:   {OUT_DIR}")


In [ ]:
# =============================================================================
# 1. DATA — Ceftazidime, E. coli, per site
# =============================================================================
site_data = {}
for s in "ABCD":
    df = pd.read_csv(DATA_ROOT / f"Proc_DRIAMS-{s}" / DRUG / "data.csv")
    df = df[df["species"] == SPECIES]
    cols = [c for c in df.columns if c.startswith("bin_")]
    site_data[s] = (df[cols].to_numpy(dtype="float32"),
                    df["label"].to_numpy(dtype="int64"))
    print(f"DRIAMS-{s}: n={len(site_data[s][1]):5d}  R={int(site_data[s][1].sum()):4d}")

X_pool = np.vstack([site_data[s][0] for s in "ABCD"])
y_pool = np.hstack([site_data[s][1] for s in "ABCD"])
print(f"pooled: n={len(y_pool)}  R={int(y_pool.sum())}")


In [ ]:
# =============================================================================
# 2. HELPERS
# =============================================================================
def tuned_threshold(params, X_tr, y_tr, seed):
    """Threshold from 3-fold out-of-fold probabilities, maximising BalAcc."""
    oof = cross_val_predict(
        RandomForestClassifier(**params, random_state=seed, n_jobs=-1),
        X_tr, y_tr, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    scores = [balanced_accuracy_score(y_tr, oof >= t) for t in THRESHOLDS]
    return float(THRESHOLDS[int(np.argmax(scores))])


def score_target(rf, X, y, threshold):
    proba = rf.predict_proba(X)[:, 1]
    return {
        "n": len(y), "n_R": int(y.sum()),
        "BalAcc@0.5": balanced_accuracy_score(y, proba >= 0.5),
        "BalAcc@tuned": balanced_accuracy_score(y, proba >= threshold),
        "AUC": roc_auc_score(y, proba),
    }


In [ ]:
# =============================================================================
# 3. SEEDED CROSS-SITE — train on 80% of A, test on A holdout + B/C/D
# =============================================================================
rows = []
conf_saved = None
for seed in SEEDS:
    idx_tr, idx_te = train_test_split(
        np.arange(len(site_data["A"][1])), test_size=0.20, stratify=site_data["A"][1],
        random_state=seed)
    state = fit_input_transform(site_data["A"][0][idx_tr], "log1p+standardize")
    X_tr = apply_input_transform(site_data["A"][0][idx_tr], state)
    y_tr = site_data["A"][1][idx_tr]

    rf = RandomForestClassifier(**PARAMS, random_state=seed, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    thr = tuned_threshold(PARAMS, X_tr, y_tr, seed)

    targets = {
        "A holdout": (apply_input_transform(site_data["A"][0][idx_te], state),
                      site_data["A"][1][idx_te]),
        "B": (apply_input_transform(site_data["B"][0], state), site_data["B"][1]),
        "C": (apply_input_transform(site_data["C"][0], state), site_data["C"][1]),
        "D": (apply_input_transform(site_data["D"][0], state), site_data["D"][1]),
    }
    targets["B+C+D"] = (
        np.concatenate([targets["B"][0], targets["C"][0], targets["D"][0]]),
        np.concatenate([targets["B"][1], targets["C"][1], targets["D"][1]]),
    )

    print(f"seed={seed:4d}  tuned threshold={thr:.2f}")
    for name, (X_t, y_t) in targets.items():
        m = score_target(rf, X_t, y_t, thr)
        rows.append({"seed": seed, "target": name, "threshold": thr, **m})
        print(f"  {name:10s} n={m['n']:5d} R={m['n_R']:4d}  "
              f"BalAcc@0.5={m['BalAcc@0.5']:.3f}  "
              f"BalAcc@tuned={m['BalAcc@tuned']:.3f}  AUC={m['AUC']:.3f}")
    if seed == 42:
        conf_saved = (rf, state)

df_cs = pd.DataFrame(rows)


In [ ]:
# =============================================================================
# 4. POOLED 75/25 — same tuned-threshold protocol
# =============================================================================
pool_rows = []
for seed in SEEDS:
    idx_tr, idx_te = train_test_split(
        np.arange(len(y_pool)), test_size=0.25, stratify=y_pool, random_state=seed)
    state = fit_input_transform(X_pool[idx_tr], "log1p+standardize")
    X_tr = apply_input_transform(X_pool[idx_tr], state)
    y_tr = y_pool[idx_tr]

    rf = RandomForestClassifier(**PARAMS, random_state=seed, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    thr = tuned_threshold(PARAMS, X_tr, y_tr, seed)
    m = score_target(rf, apply_input_transform(X_pool[idx_te], state), y_pool[idx_te], thr)
    pool_rows.append({"seed": seed, "threshold": thr, **m})
    print(f"seed={seed:4d}  tuned threshold={thr:.2f}  "
          f"BalAcc@0.5={m['BalAcc@0.5']:.4f}  "
          f"BalAcc@tuned={m['BalAcc@tuned']:.4f}  AUC={m['AUC']:.4f}")

df_pool = pd.DataFrame(pool_rows)


In [ ]:
# =============================================================================
# 5. AGGREGATE, EXPORT, FIGURES
# =============================================================================
def gstd(values):
    """Population std (ddof=0), matching report.md's np.std convention."""
    return np.std(values, ddof=0)


COLS = ["BalAcc@0.5", "BalAcc@tuned", "AUC"]
order = ["A holdout", "B", "C", "D", "B+C+D"]
print("=== Seeded cross-site (mean ± std over 5 seeds) ===")
for t in order:
    sub = df_cs[df_cs.target == t]
    print(f"{t:10s} BalAcc@0.5 {sub['BalAcc@0.5'].mean():.3f}±{gstd(sub['BalAcc@0.5'].values):.3f}  "
          f"BalAcc@tuned {sub['BalAcc@tuned'].mean():.3f}±{gstd(sub['BalAcc@tuned'].values):.3f}  "
          f"AUC {sub['AUC'].mean():.3f}±{gstd(sub['AUC'].values):.3f}")

print("\n=== Pooled 75/25 (mean ± std over 5 seeds) ===")
for col in COLS:
    print(f"{col:12s} {df_pool[col].mean():.4f} ± {gstd(df_pool[col].values):.4f}")

df_cs.to_csv(OUT_DIR / "cross_site_seeded_metrics.csv", index=False)
df_pool.to_csv(OUT_DIR / "pooled_threshold_metrics.csv", index=False)

means_cs = df_cs.groupby("target")[COLS].mean().reindex(order)
stds_cs = df_cs.groupby("target")[COLS].agg(gstd).reindex(order)
means_p = df_pool[COLS].mean()
stds_p = df_pool[COLS].agg(gstd)
colors = ["#aec7e8", "#1f77b4", "#ff7f0e"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2, 1]})
x = np.arange(len(order)); w = 0.27
for i, (col, c) in enumerate(zip(COLS, colors)):
    ax1.bar(x + (i - 1) * w, means_cs[col], w, yerr=stds_cs[col], capsize=3,
            label=col, color=c)
ax1.set_xticks(x); ax1.set_xticklabels(order)
ax1.set_ylabel("Score"); ax1.set_ylim(0, 1)
ax1.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax1.set_title("Seeded cross-site — train on A train slice (80%), 5 seeds")
ax1.legend(fontsize=8); ax1.grid(True, ls="--", lw=0.5, alpha=0.5)

xp = np.arange(3)
ax2.bar(xp, means_p.values, yerr=stds_p.values, capsize=4, color=colors)
ax2.set_xticks(xp)
ax2.set_xticklabels(["BalAcc@0.5", "BalAcc@tuned", "AUC"], rotation=15, fontsize=8)
ax2.set_ylim(0, 1); ax2.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax2.set_title("Pooled 75/25 — threshold effect (5 seeds)")
ax2.grid(True, ls="--", lw=0.5, alpha=0.5)

fig.tight_layout()
fig.savefig(OUT_DIR / "cross_site_seeded.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# =============================================================================
# 6. SEED-42 MODEL — confusion matrices on B/C/D at the 0.5 threshold
# =============================================================================
rf42, state42 = conf_saved
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, s in zip(axes, "BCD"):
    X_t = apply_input_transform(site_data[s][0], state42)
    pred = rf42.predict(X_t)
    ConfusionMatrixDisplay.from_predictions(
        site_data[s][1], pred, display_labels=["Susceptible", "Resistant"],
        cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"DRIAMS-{s} (n={len(site_data[s][1])}, predicted R={int(pred.sum())})",
                 fontsize=11)
fig.suptitle("Seed 42 — RF trained on the A train slice (0.5 threshold)",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "seeded_confusion.pdf", bbox_inches="tight")
plt.show()
